In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
import dowhy
from dowhy import CausalModel

In [2]:
root = Path().resolve().parents[0] if Path().resolve().name == "notebooks" else Path().resolve()
dfr = root / "data" / "processed" / "cleaned_data.csv"
df = pd.read_csv(dfr,low_memory=False)

In [3]:
df.head(5)

,COMPANY_ID,CONTACT_NAME,COMPANY_NAME,VEEQO_PRODUCT,ACCOUNT_OWNER_NAME,COUNTRY,SIGNUP_TYPE,EMAIL,DECILE,SELLER_SIZE,...,SHIPMENTS_BUY_SHIPPING_DHL,SHIPMENTS_BUY_SHIPPING_SWA,SHIPMENTS_BUY_SHIPPING_OTHER,SHIPMENTS_UPS,SHIPMENTS_USPS,SHIPMENTS_FEDEX,SHIPMENTS_DHL,SHIPMENTS_SWA,SHIPMENTS_OTHER,ADOBE_ECID
0,263028,King,Milu Network Tech,new product,0.0,0.0,Amazon SSO,0.0,9.0,Small,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17655601342629931110826685221210694887
1,102953,Arvind Aswani,Sporting Genius Ltd,new product,0.0,0.0,Email Sign-up,0.0,9.0,Small,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown
2,91326,Yasir,US Prime Outlet,new product,0.0,0.0,Amazon SSO,0.0,8.0,Medium,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown
3,138633,Unknown,Luxizo Depot LLC,new product,0.0,0.0,Amazon SSO,0.0,7.0,Medium,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown
4,258729,Todoroff,Cape Cod Challenger Club,new product,0.0,0.0,Amazon SSO,0.0,9.0,Small,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown


In [4]:
fba = df['TOTAL_FBA_ORDERS_SINCE_INCEPTION'].fillna(0)
total = df['TOTAL_ORDERS_SINCE_INCEPTION'].fillna(0)

df['FBA_RATIO'] = np.where(total > 0, fba / total, 0.0)
df['FBA_RATIO'] = np.clip(df['FBA_RATIO'], 0.0, 1.0)
df['FBA_RATIO'].describe()

count    84031.000000
mean         0.083305
std          0.264946
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: FBA_RATIO, dtype: float64

In [5]:
df['FBA_RATIO'].shape

In [6]:
cols = ['CALLS_15_MIN_L28', 'CALLS_2_MIN_L28', 'DEMOS_15_MIN_L28', 'DEMOS_2_MIN_L28', 'EMAILS_L28', 'RE_EMAILS']
print([(c, c in df.columns) for c in cols])

[('CALLS_15_MIN_L28', True), ('CALLS_2_MIN_L28', True), ('DEMOS_15_MIN_L28', True), ('DEMOS_2_MIN_L28', True), ('EMAILS_L28', True), ('RE_EMAILS', True)]


In [7]:
df['ACCOUNT_OWNER_NAME'].notnull().sum()

np.int64(84031)

In [8]:
sales_cols = ['CALLS_15_MIN_L28', 'CALLS_2_MIN_L28', 
              'DEMOS_15_MIN_L28', 'DEMOS_2_MIN_L28', 
              'EMAILS_L28', 'RE_EMAILS']

has_sales = (df[sales_cols] > 0).any(axis=1).sum()
print(f"Accounts with at least one sales touch: {has_sales} out of {len(df)}")

Accounts with at least one sales touch: 8987 out of 84031


In [9]:
df['HAS_SALES_TOUCH'] = (df[sales_cols] > 0).any(axis=1).astype(int)

In [10]:
confounders = ['DECILE', 'TOTAL_ORDERS_SINCE_INCEPTION', 'FBA_RATIO', 'TIME_TO_LAUNCHED']

In [11]:
treated = df[df['HAS_SALES_TOUCH'] == 1]
control = df[df['HAS_SALES_TOUCH'] == 0]

In [12]:
smd_results = {}
for col in confounders:
    mean_t = treated[col].mean()
    mean_c = control[col].mean()
    var_t = treated[col].var()
    var_c = control[col].var()
    pooled_std = np.sqrt((var_t + var_c) / 2)
    smd = (mean_t - mean_c) / pooled_std
    smd_results[col] = abs(smd)

In [18]:
smd_df = pd.DataFrame(list(smd_results.items()), columns=['Confounder', 'SMD'])
print(smd_df)

                     Confounder       SMD
0                        DECILE  1.274890
1  TOTAL_ORDERS_SINCE_INCEPTION  0.327755
2                     FBA_RATIO  0.331302
3              TIME_TO_LAUNCHED  0.324924


## selection bias exists on all four dimensions

In [21]:
possible_outcome = [c for c in df.columns if 'WON' in c.upper() or 'CONVERT' in c.upper() or 'PAID' in c.upper()]
print(possible_outcome)

[]


In [24]:
outcome = 'TOTAL_ORDERS_SINCE_INCEPTION'

mean_treated = df[df['HAS_SALES_TOUCH'] == 1][outcome].mean()
mean_control = df[df['HAS_SALES_TOUCH'] == 0][outcome].mean()
raw_gap = mean_treated - mean_control

In [25]:
print(mean_treated, mean_control, raw_gap)

30178.149994436408 1862.9041628911039 28315.245831545304


# Analysis 3

In [26]:
skew_cols = ['TOTAL_ORDERS_SINCE_INCEPTION', 'TOTAL_SHIPMENTS_SINCE_INCEPTION', 
             'TOTAL_FBA_ORDERS_SINCE_INCEPTION', 'TOTAL_ORDERS_YTD']

skew_values = {}
for col in skew_cols:
    if col in df.columns:
        skew_values[col] = df[col].skew()

skew_df = pd.DataFrame(list(skew_values.items()), columns=['Variable', 'Skewness'])
print(skew_df)

                           Variable    Skewness
0      TOTAL_ORDERS_SINCE_INCEPTION   47.356346
1   TOTAL_SHIPMENTS_SINCE_INCEPTION   91.273287
2  TOTAL_FBA_ORDERS_SINCE_INCEPTION  180.447041
3                  TOTAL_ORDERS_YTD   65.361097


## Analysis 2

In [28]:
pql = df['PQL_DATE']
demo = df['DISCOVERY_MEETING_DATE']
launch = df['LAUNCHED_DATE']
upgrade = df['UPGRADED_TO_POWER_AT']

total = len(df)

order1 = (demo > pql).sum()
order2 = (launch > demo).sum()
order3 = (upgrade > launch).sum()

skip_demo = (~demo.notna() & launch.notna()).sum()
skip_launch = (~launch.notna() & upgrade.notna()).sum()
loop = (demo > launch).sum()

print(total)
print(order1)
print(order2)
print(order3)
print(skip_demo)
print(skip_launch)
print(loop)

84031
0
0
0
0
0
0


## Analyze 4

In [29]:
cols = ['CALLS_15_MIN_L28', 'CALLS_2_MIN_L28', 'DEMOS_15_MIN_L28', 'DEMOS_2_MIN_L28', 'EMAILS_L28', 'RE_EMAILS']
print(df[cols].corr())

                  CALLS_15_MIN_L28  CALLS_2_MIN_L28  DEMOS_15_MIN_L28  \
CALLS_15_MIN_L28          1.000000         0.541745          0.272906   
CALLS_2_MIN_L28           0.541745         1.000000          0.352141   
DEMOS_15_MIN_L28          0.272906         0.352141          1.000000   
DEMOS_2_MIN_L28           0.272906         0.352141          1.000000   
EMAILS_L28                0.142053         0.229313          0.562394   
RE_EMAILS                 0.018585         0.033869          0.142523   

                  DEMOS_2_MIN_L28  EMAILS_L28  RE_EMAILS  
CALLS_15_MIN_L28         0.272906    0.142053   0.018585  
CALLS_2_MIN_L28          0.352141    0.229313   0.033869  
DEMOS_15_MIN_L28         1.000000    0.562394   0.142523  
DEMOS_2_MIN_L28          1.000000    0.562394   0.142523  
EMAILS_L28               0.562394    1.000000   0.211358  
RE_EMAILS                0.142523    0.211358   1.000000  


## Conclusion